# Evaluating Natural Language Explanations of Food Recommendation System XFoodRec Across Several LLMs

Authors: Junxuan Ling, Tianyi Liu, Anqi Zhou

**Connect to google drive persistent storage**

Git workflow and persistent-storage setup are documented in `collaboration_setup.md`

In [78]:
# load from drive
from google.colab import drive
drive.mount("/content/drive")

# 1. Data Exploration & Preparation

## 1.1. Load and Inspect

### a. Load Data

In [79]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


data_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data/raw")
persona_path = data_dir / "personas.json"
recommendation_path = data_dir / "recommendations.json"
recipe_path = data_dir / "recipes.parquet"

In [80]:
# open files and load data

with open(persona_path, "r", encoding="utf-8") as f:
    personas = json.load(f)

with open(recommendation_path, "r", encoding="utf-8") as f:
    recommendations = json.load(f)

recipes = pd.read_parquet(recipe_path)

### b. Inspect Raw data

import helper functions

In [81]:
%pip install -q spacy pyarrow nltk
!python -m spacy download en_core_web_sm

import subprocess
import sys

# because I use Colab remote kernel, to refer to the preprocess_helpers.py, have to 
# clone the github repo to find the .py file
repo_url = "https://github.com/lay-animals-projects/food-recommender-evaluation.git"
repo_branch = "anqizhou"
repo_dir = Path("/content/food-recommender-evaluation")

if (repo_dir / ".git").exists():
    subprocess.run(
        ["git", "-C", str(repo_dir), "fetch", "origin", repo_branch],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo_dir), "checkout", repo_branch],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", repo_branch],
        check=True,
    )
elif repo_dir.exists():
    raise FileExistsError(
        f"{repo_dir} exists but is not a Git repository."
    )
else:
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "--branch", repo_branch, repo_url, str(repo_dir),
        ],
        check=True,
    )

helper_dir = repo_dir / "notebooks"
helper_path = helper_dir / "preprocess_helpers.py"

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from preprocess_helpers import (
    build_text_records,
    build_matching_resources,
    color_for_combination,
    compute_coverage,
    contains_sequence,
    explanation_words,
    focus_combination,
    is_empty_value,
    make_lemma_table,
    make_token_table,
    term_matches,
)


In [82]:
print("personas")
print("type:", type(personas))
print("length:", len(personas))

print(json.dumps(personas[0],indent=2))

print("\nrecommendations")
print("type:", type(recommendations))
print("length:", len(recommendations))

print(json.dumps(recommendations[0],indent=2))

print("\nrecipes")
print("type:", type(recipes))
print("shape:", recipes.shape)

display(recipes.head())


### c. Organize Raw Data as DataFrames and Further Inspect

In [85]:
# pd display setting
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

#### c.1. persona

In [86]:
personas_df = pd.json_normalize(personas)

print(personas_df.shape)
display(personas_df.head())

In [87]:
personas_df.info()

In [88]:
display(pd.DataFrame({
    "missing": personas_df.isna().sum(),
    "empty_list_or_text": personas_df.apply(lambda col: col.map(is_empty_value).sum())
}))

In [89]:
print("unique personas:", personas_df["id"].nunique())
print("duplicate ids:", personas_df["id"].duplicated().sum())

In [90]:
# persona distribution

distribution_columns = {
    "Gender": "profile.gender",
    "Dietary Goal": "profile.dietary_goal",
    "Activity Level": "profile.activityLevel",
    "Dietary Restriction": "profile.dietaryProfile.dietaryRestrictions.selected",
    "Food Allergy": "profile.dietaryProfile.foodAllergies.selected",
    "Health Condition": "profile.dietaryProfile.healthConditions.selected"
}

persona_distribution = {}

for name, column in distribution_columns.items():
    values = personas_df[column]

    if values.apply(lambda x: isinstance(x, list)).all():
        values = values.explode()

    counts = values.value_counts()
    persona_distribution[name] = pd.Series(
        [f"{category}: {count}" for category, count in counts.items()]
    )

display(pd.DataFrame(persona_distribution).fillna(""))

# Show no selection separately from missing dietary-restriction data.
restriction_values = personas_df["profile.dietaryProfile.dietaryRestrictions.selected"]
display(restriction_values.map(
    lambda x: ["No selection"] if isinstance(x, list) and not x else x
).explode().fillna("Missing").value_counts())


#### c.2. recommendation

In [91]:
# rank is the position in the original recommendation list (store rank for future tasks).
# persona -> recommendations

recommendation_rows = []

for item in recommendations:
    persona_id = item["persona"]["id"]

    for rank, rec in enumerate(item["recommendations"], start=1):
        recommendation_rows.append({
            **rec,
            "persona_id": persona_id,
            "rank": rank
        })

recommendations_df = pd.DataFrame(recommendation_rows)

print(recommendations_df.shape)
display(recommendations_df.head())

recommendations_df.info()

print(recommendations_df.columns.tolist())

print("total recommendations:", len(recommendations_df))
print("personas:", recommendations_df["persona_id"].nunique())
print("unique recipes:", recommendations_df["recipe_id"].nunique())


In [95]:
# recommentation number for personas
# include personas with zero recommendations also
rec_counts = recommendations_df.groupby("persona_id").size().reindex(personas_df["id"], fill_value=0)
display(rec_counts.describe())

display(rec_counts.sort_values())


In [98]:
pair_columns = ["persona_id", "recipe_id"]
duplicate_recs = recommendations_df.duplicated(subset=pair_columns, keep=False)

print("rows belonging to repeated persona-recipe pairs:", duplicate_recs.sum())
print(
    "repeated persona-recipe pairs:",
    recommendations_df.loc[duplicate_recs, pair_columns].drop_duplicates().shape[0],
)
print(
    "extra rows beyond one per persona-recipe pair:",
    recommendations_df.duplicated(subset=pair_columns).sum(),
)
print(
    "unique persona-recipe pairs:",
    recommendations_df[pair_columns].drop_duplicates().shape[0],
)

if duplicate_recs.any():
    display(
        recommendations_df.loc[
            duplicate_recs,
            ["persona_id", "rank", "recipe_id", "title", "explanation"],
        ].sort_values(["persona_id", "recipe_id"])
    )


#### c.3. recipe

In [101]:
print(recommendations_df["recipe_id"].dtype)
print(recipes["recipe_id"].dtype)

In [102]:
# join 2 df

recommendations_df["recipe_id"] = recommendations_df["recipe_id"].astype(str)
recipes["recipe_id"] = recipes["recipe_id"].astype(str)

matched = recommendations_df["recipe_id"].isin(recipes["recipe_id"])

print("matched:", matched.sum())
print("unmatched:", (~matched).sum())
print("match rate:", matched.mean())

In [103]:
nutrition_df = pd.json_normalize(recommendations_df["nutrition"])
nutrition_df.index = recommendations_df.index

# recommendation protein/carbs/fat are stored as strings such as "13.3g".

for column in ["calories", "protein", "carbs", "fat"]:
    values = nutrition_df[column].astype("string").str.strip()
    if column != "calories":
        values = values.str.replace(r"\s*g$", "", regex=True)
    nutrition_df[column] = pd.to_numeric(values, errors="coerce")

print("missing or unparseable recommendation nutrients:")
display(nutrition_df.isna().sum())
display(nutrition_df.head())

In [61]:
recommendations_prepared_df = pd.concat(
    [recommendations_df.drop(columns="nutrition"), nutrition_df.add_prefix("rec_")],
    axis=1
)

In [62]:
# many recommendations may reference one source recipe.

merged_df = recommendations_prepared_df.merge(
    recipes,
    on="recipe_id",
    how="left",
    suffixes=("_rec", "_recipe"),
    indicator=True,
    validate="many_to_one"
)

print(merged_df.shape)
display(merged_df.head())

In [63]:
# check if all matched successfully

print(
    "recommendations without recipe data:",
    merged_df["_merge"].eq("left_only").sum()
)

## 1.2. Extract all explanations and compute basic statistics

### a. Extract all explanations

In [64]:
import re

explanations_df = recommendations_df[
    ["persona_id", "recipe_id", "rank", "title", "explanation"]
].copy()

display(explanations_df.head())


### b. Compute Length (words/characters)

In [65]:
# add length

word_pattern = re.compile(r"\b\w+(?:[-'’]\w+)*\b") # to filter non-standard texts

explanations_df["word_count"] = explanations_df["explanation"].str.findall(word_pattern).str.len()
explanations_df["character_count"] = explanations_df["explanation"].str.len()

display(explanations_df.head())

In [66]:
# distribution

analysis_df = explanations_df.copy()

length_summary = analysis_df[["word_count", "character_count"]].describe().T.rename(
    columns={"50%": "median"}
)
display(length_summary.round(2))

In [67]:
# plot

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, column, label in zip(axes, ["word_count", "character_count"], ["Words", "Characters"]):
    ax.hist(analysis_df[column], bins=20, edgecolor="white")
    ax.set(xlabel=label, ylabel="Recommendation entries", title=f"Explanation length ({label.lower()})")
plt.tight_layout()
plt.show()

### b-c-1 Preprocessing raw data for further analysis

 (Run ONLY Once, and store the result for future use)

compose records for recommendation explanations, recipes and personas (raw text for now).

In [69]:
explanation_records, recipe_records, persona_records = build_text_records(
    recommendations,
    recipes,
    personas_df,
)


compose lemmatized text of the former records

In [70]:
explanation_lemmas_df = make_lemma_table(explanation_records)
recipe_lemmas_df = make_lemma_table(recipe_records)
persona_lemmas_df = make_lemma_table(persona_records)

explanation_lemmas_df["rank"] = (
    explanation_lemmas_df["rank"].astype("Int64")
)

assert explanation_lemmas_df["recommendation_id"].is_unique
assert len(explanation_lemmas_df) == len(recommendations_df)

save results for future use

In [71]:
# store the processed text for future use
processed_dir = Path(
    "/content/drive/MyDrive/XFoodRec_Project_Data/processed"
)
processed_dir.mkdir(parents=True, exist_ok=True)


explanation_lemmas_df.to_parquet(
    processed_dir / "explanation_lemmas.parquet",
    index=False
)
recipe_lemmas_df.to_parquet(
    processed_dir / "recipe_lemmas.parquet",
    index=False
)
persona_lemmas_df.to_parquet(
    processed_dir / "persona_lemmas.parquet",
    index=False
)

print("Saved processed files to:", processed_dir)
print("Explanation records:", len(explanation_lemmas_df))
print("Recipe text items:", len(recipe_lemmas_df))
print("Persona text items:", len(persona_lemmas_df))

display(
    explanation_lemmas_df[
        [
            "recommendation_id",
            "persona_id",
            "recipe_id",
            "rank",
            "explanation_lemmas"
        ]
    ].head()
)

display(
    recipe_lemmas_df.loc[
        recipe_lemmas_df["source_field"].eq("recipe_ingredient"),
        ["recipe_id", "recipe_text_lemmas"]
    ].head()
)

Save token tables and nutrition field names

In [72]:
# build the three token tables from the same records as the lemma tables.
recommendation_tokens_df = make_token_table(explanation_records)
recipe_tokens_df = make_token_table(recipe_records)
persona_tokens_df = make_token_table(persona_records)

for filename, table in [
    ("recommendation_tokens.parquet", recommendation_tokens_df),
    ("recipe_tokens.parquet", recipe_tokens_df),
    ("persona_tokens.parquet", persona_tokens_df),
]:
    table.to_parquet(processed_dir / filename, index=False)
    print(f"Saved {filename}: {len(table)} text items")

display(recommendation_tokens_df[["recommendation_id", "text", "tokens"]].head())

In [73]:
# collect actual nutrition keys rather than maintaining a separate word list.
nutrition_fields = list(dict.fromkeys(
    field
    for item in recommendations
    for recommendation in item["recommendations"]
    for field in recommendation["nutrition"]
))
nutrition_fields_path = processed_dir / "nutrition_fields.json"
with nutrition_fields_path.open("w", encoding="utf-8") as f:
    json.dump(nutrition_fields, f, ensure_ascii=False, indent=2)

print("Saved nutrition fields:", nutrition_fields)

### b-c-2 reload preprocessed data

In [74]:
processed_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data/processed")

explanation_lemmas_df = pd.read_parquet(processed_dir / "explanation_lemmas.parquet")
recipe_lemmas_df = pd.read_parquet(processed_dir / "recipe_lemmas.parquet")
persona_lemmas_df = pd.read_parquet(processed_dir / "persona_lemmas.parquet")
nutrition_fields_path = processed_dir / "nutrition_fields.json"

print("Explanations:", len(explanation_lemmas_df))
print("Recipe text items:", len(recipe_lemmas_df))
print("Persona text items:", len(persona_lemmas_df))


Persisted file schemas and shared-helper behavior are documented in `data_exploration_readme.md`

In [75]:
recipe_table = recipes
with open(nutrition_fields_path, "r", encoding="utf-8") as f:
    nutrition_fields = json.load(f)
matching_resources = build_matching_resources(persona_lemmas_df, recipe_lemmas_df, recipe_table, nutrition_fields)
print("Nutrients recorded in the data:", matching_resources["nutrient_terms"])
print("Health label values (goals, diets, conditions):", len(matching_resources["health_label_values"]))
print("Personas with values:", len(matching_resources["persona_values"]))
print("Recipes with ingredient names:", len(matching_resources["ingredient_names"]),
      "| food words:", len(matching_resources["food_terms"]))

### c. Distribution of explanation types (ingredient-, nutrition-, taste-focused)

#### c.1. Classify explanations by focus


In [ ]:
nutrition_terms = set(matching_resources["nutrient_terms"])
taste_terms = {
    "taste", "flavor", "flavour", "delicious", "tasty",
    "savory", "savoury", "sweet", "spicy", "tangy",
    "creamy", "crunchy", "crispy", "tender", "juicy",
    "aromatic", "refreshing", "hearty", "satisfy",
    "comfort", "warm", "mild", "bitter", "sour", "smoky",
    "texture",
}

analysis_df = explanation_lemmas_df.copy()
ingredient_matches = []
nutrition_matches = []
taste_matches = []

for row in analysis_df.to_dict("records"):
    explanation_body = explanation_words(row, matching_resources)
    explanation_terms = set(explanation_body)
    recipe_ingredients = matching_resources["ingredient_names"].get(
        str(row["recipe_id"]),
        set(),
    )

    ingredient_matches.append(sorted(
        " ".join(ingredient)
        for ingredient in recipe_ingredients
        if contains_sequence(explanation_body, list(ingredient))
    ))
    nutrition_matches.append(sorted(
        nutrient
        for nutrient in nutrition_terms
        if any(term_matches(term, nutrient) for term in explanation_body)
    ))
    taste_matches.append(sorted(explanation_terms & taste_terms))

analysis_df["ingredient_matches"] = ingredient_matches
analysis_df["nutrition_matches"] = nutrition_matches
analysis_df["taste_matches"] = taste_matches
analysis_df["ingredient_focused"] = analysis_df["ingredient_matches"].map(bool)
analysis_df["nutrition_focused"] = analysis_df["nutrition_matches"].map(bool)
analysis_df["taste_focused"] = analysis_df["taste_matches"].map(bool)

focus_columns = [
    "ingredient_focused",
    "nutrition_focused",
    "taste_focused",
]
focus_labels = {
    "ingredient_focused": "Ingredient",
    "nutrition_focused": "Nutrition",
    "taste_focused": "Taste",
}

total_explanations = len(analysis_df)
if total_explanations == 0:
    raise ValueError("No explanation records were loaded.")
if not analysis_df[focus_columns].any(axis=None):
    raise ValueError(
        "No focus matches were found. Check the loaded lemmas and vocabularies."
    )


#### c.2. Summarize focus categories


In [ ]:
type_summary = pd.DataFrame({
    "count": analysis_df[focus_columns].sum().astype(int),
    "percent": analysis_df[focus_columns].mean().mul(100).round(1),
})
type_summary.index = [focus_labels[column] for column in type_summary.index]
display(type_summary)

analysis_df["focus_combination"] = analysis_df.apply(
    focus_combination,
    axis=1,
)
combination_summary = (
    analysis_df["focus_combination"]
    .value_counts()
    .rename_axis("focus_combination")
    .to_frame("count")
)
combination_summary["percent"] = (
    combination_summary["count"]
    .div(total_explanations)
    .mul(100)
    .round(1)
)
display(combination_summary)


#### c.3. Visualize focus categories


In [ ]:
from matplotlib.patches import Patch

focus_colors = {
    "Ingredient": "#E76F51",
    "Nutrition": "#2A9D8F",
    "Taste": "#E9C46A",
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    gridspec_kw={"width_ratios": [0.8, 1.5]},
)

left_labels = ["Ingredient", "Nutrition", "Taste"]
left_counts = [int(analysis_df[column].sum()) for column in focus_columns]
left_percentages = [100 * count / total_explanations for count in left_counts]
bars = axes[0].bar(
    left_labels,
    left_counts,
    color=[focus_colors[label] for label in left_labels],
)
axes[0].bar_label(
    bars,
    labels=[
        f"{count} ({percentage:.1f}%)"
        for count, percentage in zip(left_counts, left_percentages)
    ],
    padding=3,
)
axes[0].set_title("Explanations by focus")
axes[0].set_ylabel("Number of explanations")
axes[0].set_ylim(0, max(left_counts) * 1.18)
axes[0].tick_params(axis="x", rotation=15)

plot_data = combination_summary.sort_values("count")
combination_labels = plot_data.index.tolist()
combination_counts = plot_data["count"].tolist()
combination_percentages = plot_data["percent"].tolist()
bars = axes[1].barh(
    combination_labels,
    combination_counts,
    color=[color_for_combination(label) for label in combination_labels],
)
axes[1].bar_label(
    bars,
    labels=[
        f"{count} ({percentage:.1f}%)"
        for count, percentage in zip(combination_counts, combination_percentages)
    ],
    padding=3,
)
axes[1].set_title("Exact focus combinations")
axes[1].set_xlabel("Number of explanations")
axes[1].set_xlim(0, max(combination_counts) * 1.35)
axes[1].legend(
    handles=[
        Patch(facecolor=color, label=label)
        for label, color in focus_colors.items()
    ],
    title="Focus colors",
    loc="lower right",
)

fig.suptitle(f"Explanation focus distribution (n={total_explanations})")
plt.tight_layout()
plt.show()


#### c.4. Inspect matched terms


In [ ]:
display(
    analysis_df[
        [
            "recommendation_id",
            "ingredient_matches",
            "nutrition_matches",
            "taste_matches",
            "focus_combination",
        ]
    ].head(15)
)


### d. Coverage of user preferences vs. health justification

Each explanation is checked for two things: whether it refers to the user's preferences, and whether it gives a health justification. One explanation can count as both, one, or neither. Only things that exist in the data count.

| What the explanation mentions | Counted as |
|---|---|
| The user's own liked or disliked ingredients or favourite cuisines (e.g. "lentils, which you enjoy", or "features chicken" for a user who likes chicken) | Preference |
| A nutrient recorded in the recipe data, with or without numbers (e.g. "high in protein", "33.1 g of protein", "low in sodium") | Health |
| A goal, diet or health condition from the user profiles (e.g. "weight loss", "vegan", "low sodium") | Health |
| An allergen, only for the user who has that allergy | Health |
| General wording that names none of the above (e.g. "fits your preferences", "suits your dietary needs", "healthy") | Neither (left to the transparency analysis) |

The recipe title is removed from the explanation before matching, so words that only appear in the title are not counted.

In [ ]:
coverage_df = compute_coverage(
    explanation_lemmas_df,
    matching_resources,
)

existing_coverage_columns = [
    column
    for column in coverage_df.columns
    if column != "recommendation_id" and column in analysis_df.columns
]
analysis_df = (
    analysis_df
    .drop(columns=existing_coverage_columns)
    .merge(
        coverage_df,
        on="recommendation_id",
        how="left",
        validate="one_to_one",
    )
)


#### d.1. Summarize coverage


In [ ]:
total_explanations = len(coverage_df)
coverage_order = [
    "Both",
    "Preference only",
    "Health only",
    "Neither",
]
coverage_summary = (
    coverage_df["coverage_group"]
    .value_counts()
    .reindex(coverage_order, fill_value=0)
    .rename("count")
    .to_frame()
)
coverage_summary["percent"] = (
    100 * coverage_summary["count"] / total_explanations
).round(1)
display(coverage_summary)

has_nutrition_evidence = coverage_df["nutrition_evidence"].map(bool)
has_health_evidence = coverage_df["health_evidence"].map(bool)
evidence_summary = pd.DataFrame(
    {
        "explanations": [
            coverage_df["preference_covered"].sum(),
            coverage_df["health_justification"].sum(),
            has_nutrition_evidence.sum(),
            has_health_evidence.sum(),
        ]
    },
    index=[
        "Preference: user's own ingredient or cuisine",
        "Health justification (any)",
        "  nutrient recorded in the recipe data",
        "  goal, diet, allergy or condition",
    ],
)
evidence_summary["percent"] = (
    100 * evidence_summary["explanations"] / total_explanations
).round(1)
display(evidence_summary)


#### d.2. Visualize coverage


In [ ]:
coverage_colors = {
    "Both": "#2A9D8F",
    "Preference only": "#E9C46A",
    "Health only": "#E76F51",
    "Neither": "#B8B8B8",
}
plot_order = coverage_order[::-1]
plot_counts = coverage_summary.loc[plot_order, "count"]
plot_colors = [coverage_colors[group] for group in plot_order]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(plot_order, plot_counts, color=plot_colors)
ax.bar_label(bars, padding=3)
ax.set_xlabel("Number of explanations")
ax.set_title(
    "Coverage of user preferences vs. health justification "
    f"(n={total_explanations})"
)
ax.set_xlim(0, coverage_summary["count"].max() * 1.15)
plt.tight_layout()
plt.show()
